# MedSAM2 Video Evaluation — Zero-Shot vs ARCADE v2 Fine-Tuned
**Dataset:** CoronaryDominance (2 studies)  
**Comparison:** Original MedSAM2 (zero-shot) vs ARCADE v2 fine-tuned checkpoint  
**Prompt:** Single centroid click on auto-selected prompt frame

## 1 — Setup

In [ ]:
import os, subprocess, sys

subprocess.run(['sudo','apt-get','update','-q'], check=True)
subprocess.run(['sudo','apt-get','install','-y','ffmpeg','p7zip-full'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'torch==2.5.1','torchvision==0.20.1',
    '--index-url','https://download.pytorch.org/whl/cu121','-q'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'numpy>=2.0.1','opencv-python','pillow','matplotlib','tqdm',
    'hydra-core','omegaconf','iopath','-q'], check=True)

if not os.path.exists('/home/jupyter/MedSAM2'):
    subprocess.run(['git','clone','https://github.com/bowang-lab/MedSAM2.git',
                    '/home/jupyter/MedSAM2'], check=True)

sys.path.insert(0, '/home/jupyter/MedSAM2')
print('Environment ready')

## 2 — Config & Paths

In [ ]:
import os

BUCKET      = 'gs://YOUR_GCS_BUCKET'  # <-- set your bucket
CONFIG      = 'configs/sam2.1_hiera_t512.yaml'
ZS_CKPT     = '/home/jupyter/MedSAM2/checkpoints/MedSAM2_latest.pt'
FT_CKPT     = '/home/jupyter/medsam2_arcade_v2.pt'
CD_DIR      = '/tmp/coronary_dominance'
FRAMES_1    = '/home/jupyter/cd_study1_frames'
FRAMES_2    = '/home/jupyter/cd_study2_frames'
OUT_DIR     = '/home/jupyter/video_eval'

for d in [CD_DIR, FRAMES_1, FRAMES_2, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Pull fine-tuned checkpoint from GCS if missing
if not os.path.exists(FT_CKPT):
    print('Pulling ARCADE v2 checkpoint from GCS...')
    subprocess.run(['gsutil','cp',
        f'{BUCKET}/checkpoints/medsam2_arcade_v2.pt', FT_CKPT], check=True)

# Pull zero-shot checkpoint if missing
if not os.path.exists(ZS_CKPT):
    print('Pulling MedSAM2_latest.pt from GCS...')
    subprocess.run(['gsutil','cp',
        f'{BUCKET}/checkpoints/MedSAM2_latest.pt', ZS_CKPT], check=True)

print(f'Zero-shot : {ZS_CKPT}  exists={os.path.exists(ZS_CKPT)}')
print(f'Fine-tuned: {FT_CKPT}  exists={os.path.exists(FT_CKPT)}')

## 3 — Pull & Extract CoronaryDominance Data

In [ ]:
import glob, re
from collections import defaultdict

# Download & extract only if not already on disk
if not glob.glob(f'{CD_DIR}/**/*.npz', recursive=True):
    print('Downloading domain_shift_dataset.7z from GCS (~7.8 GB)...')
    subprocess.run(['gsutil','cp',
        f'{BUCKET}/domain_shift_dataset.7z',
        '/tmp/domain_shift_dataset.7z'], check=True)
    print('Extracting...')
    subprocess.run(['7z','x','/tmp/domain_shift_dataset.7z',
        f'-o{CD_DIR}/','-y'], check=True)
    os.remove('/tmp/domain_shift_dataset.7z')
    print('Done')
else:
    print('Data already on disk')

npz_all = sorted(glob.glob(f'{CD_DIR}/**/*.npz', recursive=True))
print(f'Total NPZ files: {len(npz_all)}')

# Group by study — pick one NPZ (largest) per study for 2 diverse studies
study_map = defaultdict(list)
for f in npz_all:
    m = re.search(r'(Study\w+)', f)
    study_id = m.group(1) if m else os.path.basename(os.path.dirname(os.path.dirname(f)))
    study_map[study_id].append(f)

studies = sorted(study_map.keys())
NPZ_1 = max(study_map[studies[0]], key=os.path.getsize)
NPZ_2 = max(study_map[studies[1]], key=os.path.getsize)

print(f'Study 1: {os.path.basename(NPZ_1)}  ({os.path.getsize(NPZ_1)/1e6:.0f} MB)')
print(f'Study 2: {os.path.basename(NPZ_2)}  ({os.path.getsize(NPZ_2)/1e6:.0f} MB)')

## 4 — Extract Frames & Inspect

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def extract_frames(npz_path, frames_dir):
    existing = sorted(glob.glob(f'{frames_dir}/*.jpeg'))
    if existing:
        print(f'{frames_dir}: {len(existing)} frames already extracted')
        return existing

    data = np.load(npz_path, allow_pickle=True)
    print(f'NPZ keys: {list(data.keys())}')

    frame_array = None
    for key in ['imgs', 'frames', 'images', 'data', 'video']:
        if key in data:
            frame_array = data[key]
            print(f'  key="{key}"  shape={frame_array.shape}  dtype={frame_array.dtype}')
            break
    if frame_array is None:
        frame_array = list(data.values())[0]
        print(f'  fallback first value  shape={frame_array.shape}')

    saved = []
    for i, frame in enumerate(frame_array):
        if frame.max() <= 1.0:
            frame = (frame * 255).astype(np.uint8)
        else:
            frame = frame.astype(np.uint8)
        if frame.ndim == 2:
            frame = np.stack([frame]*3, axis=-1)
        elif frame.ndim == 3 and frame.shape[0] in [1, 3]:
            frame = np.transpose(frame, (1, 2, 0))
            if frame.shape[2] == 1:
                frame = np.repeat(frame, 3, axis=2)
        path = f'{frames_dir}/{i:05d}.jpeg'
        Image.fromarray(frame).save(path, quality=95)
        saved.append(path)

    print(f'Saved {len(saved)} frames → {frames_dir}')
    return saved

print('=== Study 1 ===')
study1_frames = extract_frames(NPZ_1, FRAMES_1)
print('=== Study 2 ===')
study2_frames = extract_frames(NPZ_2, FRAMES_2)

# Show first frame of each study
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, frames, label in [
    (axes[0], study1_frames, 'Study 1'),
    (axes[1], study2_frames, 'Study 2'),
]:
    ax.imshow(np.array(Image.open(frames[0])), cmap='gray')
    ax.set_title(f'{label} — Frame 0  ({len(frames)} total frames)')
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/study_first_frames.png', dpi=120)
plt.show()

## 5 — Auto-Select Prompt Frame & Centroid Click

In [ ]:
def auto_prompt(frame_files):
    """Select highest-variance frame as prompt; centroid of brightest
    center-weighted pixels as click coordinate."""
    variances = []
    for f in frame_files:
        g = np.array(Image.open(f).convert('L'), dtype=np.float32)
        variances.append(g.var())

    prompt_frame = int(np.argmax(variances))

    gray = np.array(Image.open(frame_files[prompt_frame]).convert('L'), dtype=np.float32)
    H, W = gray.shape

    # Gaussian center-weight: down-weight bright border artifacts
    ys, xs = np.mgrid[0:H, 0:W]
    sigma  = min(H, W) / 3.0
    cw     = np.exp(-((xs - W/2)**2 + (ys - H/2)**2) / (2 * sigma**2))
    weighted = gray * cw

    thresh = np.percentile(weighted[weighted > 0], 85)
    bright_ys, bright_xs = np.where(weighted > thresh)

    click_x = int(np.mean(bright_xs))
    click_y = int(np.mean(bright_ys))
    return prompt_frame, click_x, click_y

pf1, cx1, cy1 = auto_prompt(study1_frames)
pf2, cx2, cy2 = auto_prompt(study2_frames)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, frames, pf, cx, cy, label in [
    (axes[0], study1_frames, pf1, cx1, cy1, 'Study 1'),
    (axes[1], study2_frames, pf2, cx2, cy2, 'Study 2'),
]:
    ax.imshow(np.array(Image.open(frames[pf])), cmap='gray')
    ax.plot(cx, cy, 'r+', markersize=24, markeredgewidth=3)
    ax.set_title(f'{label} — Prompt frame {pf}, centroid click ({cx}, {cy})')
    ax.axis('off')

plt.suptitle('Auto-selected prompt frames and centroid click points', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/prompt_selection.png', dpi=120)
plt.show()

print(f'Study 1: prompt_frame={pf1}  click=({cx1},{cy1})')
print(f'Study 2: prompt_frame={pf2}  click=({cx2},{cy2})')

## 6 — Load Models

In [ ]:
import torch
from sam2.build_sam import build_sam2_video_predictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "n/a"}')

# Zero-shot: original MedSAM2 weights, no fine-tuning
print('\nLoading zero-shot predictor (original MedSAM2)...')
zs_predictor = build_sam2_video_predictor(CONFIG, ZS_CKPT, device=device)
print('Zero-shot ready')

# Fine-tuned: same architecture, ARCADE v2 weights injected
print('\nLoading fine-tuned predictor (ARCADE v2)...')
ft_predictor = build_sam2_video_predictor(CONFIG, ZS_CKPT, device=device)
ft_state = torch.load(FT_CKPT, map_location=device, weights_only=False)
missing, unexpected = ft_predictor.load_state_dict(ft_state, strict=False)
ft_predictor.eval()
print(f'ARCADE v2 ready  |  missing={len(missing)}  unexpected={len(unexpected)}')

## 7 — Run Inference (Both Models, Both Studies)

In [ ]:
def run_video_inference(predictor, frames_dir, prompt_frame, click_x, click_y):
    """Propagate mask forward and backward from prompt frame.
    Returns dict {frame_idx: bool_mask_HW}."""
    masks = {}
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        state = predictor.init_state(video_path=frames_dir)
        predictor.add_new_points_or_box(
            inference_state=state,
            frame_idx=prompt_frame,
            obj_id=1,
            points=np.array([[click_x, click_y]], dtype=np.float32),
            labels=np.array([1], dtype=np.int32),
        )
        for out_idx, _, out_logits in predictor.propagate_in_video(state):
            masks[out_idx] = (out_logits[0, 0] > 0.0).cpu().numpy()
    return masks

print('=== Study 1 ===')
print('  Zero-shot inference...')
zs_masks_1 = run_video_inference(zs_predictor, FRAMES_1, pf1, cx1, cy1)
print('  Fine-tuned inference...')
ft_masks_1 = run_video_inference(ft_predictor, FRAMES_1, pf1, cx1, cy1)
print(f'  Done: {len(zs_masks_1)} frames propagated')

print('\n=== Study 2 ===')
print('  Zero-shot inference...')
zs_masks_2 = run_video_inference(zs_predictor, FRAMES_2, pf2, cx2, cy2)
print('  Fine-tuned inference...')
ft_masks_2 = run_video_inference(ft_predictor, FRAMES_2, pf2, cx2, cy2)
print(f'  Done: {len(zs_masks_2)} frames propagated')

## 8 — Frame-by-Frame Comparison Grid

In [ ]:
def add_label_bar(ax, text, color):
    """Overlay a coloured label strip at the top of an axes."""
    ax.text(0.5, 0.97, text, transform=ax.transAxes,
            fontsize=8, fontweight='bold', color='white',
            ha='center', va='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85, linewidth=0))

def comparison_grid(frame_files, zs_masks, ft_masks, prompt_frame, title, save_path, n_cols=10):
    indices = np.linspace(0, len(frame_files)-1, n_cols, dtype=int)
    fig, axes = plt.subplots(3, n_cols, figsize=(3.6*n_cols, 11))

    row_colors = ['#333333', '#c0392b', '#2471a3']
    row_labels  = ['Original', 'Zero-Shot MedSAM2', 'ARCADE v2 Fine-Tuned']

    for col, idx in enumerate(indices):
        frame = np.array(Image.open(frame_files[idx]))
        zs    = zs_masks.get(idx, np.zeros(frame.shape[:2], dtype=bool))
        ft    = ft_masks.get(idx, np.zeros(frame.shape[:2], dtype=bool))
        star  = '★ ' if idx == prompt_frame else ''

        axes[0][col].imshow(frame, cmap='gray')
        axes[0][col].set_title(f'{star}Fr.{idx}', fontsize=9)
        axes[0][col].axis('off')

        axes[1][col].imshow(frame, cmap='gray')
        axes[1][col].imshow(zs, alpha=0.45, cmap='Reds')
        axes[1][col].axis('off')

        axes[2][col].imshow(frame, cmap='gray')
        axes[2][col].imshow(ft, alpha=0.45, cmap='Blues')
        axes[2][col].axis('off')

        # Label bar on every panel (so it's visible even without the ylabel)
        for row, (color, label) in enumerate(zip(row_colors, row_labels)):
            add_label_bar(axes[row][col], label, color)

    # Also keep left-side ylabels for accessibility
    for row, label in enumerate(row_labels):
        axes[row][0].set_ylabel(label, fontsize=11, rotation=90, labelpad=8)

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

comparison_grid(
    study1_frames, zs_masks_1, ft_masks_1, pf1,
    'Study 1 — Zero-Shot vs ARCADE v2 Fine-Tuned  (★ = prompt frame)',
    f'{OUT_DIR}/study1_frame_comparison.png'
)

comparison_grid(
    study2_frames, zs_masks_2, ft_masks_2, pf2,
    'Study 2 — Zero-Shot vs ARCADE v2 Fine-Tuned  (★ = prompt frame)',
    f'{OUT_DIR}/study2_frame_comparison.png'
)

subprocess.run(['gsutil','-m','cp',
    f'{OUT_DIR}/study1_frame_comparison.png',
    f'{OUT_DIR}/study2_frame_comparison.png',
    f'{BUCKET}/results/video_eval/'], check=True)
print('Frame grids saved to GCS')

## 9 — Comparison Video

In [ ]:
import cv2

def draw_label(panel, text, color_bgr):
    """White bold text on a coloured pill in the top-left corner."""
    font       = cv2.FONT_HERSHEY_DUPLEX
    scale      = 0.55
    thickness  = 1
    pad        = 6
    (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
    x1, y1     = 8, 8
    x2, y2     = x1 + tw + pad*2, y1 + th + pad*2
    cv2.rectangle(panel, (x1, y1), (x2, y2), color_bgr, -1)
    cv2.rectangle(panel, (x1, y1), (x2, y2), (255,255,255), 1)
    cv2.putText(panel, text, (x1+pad, y2-pad), font, scale, (255,255,255), thickness, cv2.LINE_AA)

def draw_frame_counter(panel, frame_idx, total, prompt_frame):
    """Frame number + prompt indicator in bottom-right."""
    text  = f'Fr {frame_idx}/{total-1}' + (' ★' if frame_idx == prompt_frame else '')
    font  = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.45
    (tw, th), _ = cv2.getTextSize(text, font, scale, 1)
    H, W  = panel.shape[:2]
    x, y  = W - tw - 8, H - 8
    cv2.putText(panel, text, (x, y), font, scale, (200,200,200), 1, cv2.LINE_AA)

def make_comparison_video(frame_files, zs_masks, ft_masks, prompt_frame, out_path, fps=12):
    """3-panel side-by-side MP4 with panel labels and frame counter."""
    sample = np.array(Image.open(frame_files[0]).convert('RGB'))
    H, W   = sample.shape[:2]

    # Divider width between panels
    div = 2
    total_w = W*3 + div*2

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_path, fourcc, fps, (total_w, H))

    n = len(frame_files)
    for i, path in enumerate(frame_files):
        frame = np.array(Image.open(path).convert('RGB'))
        zs    = zs_masks.get(i, np.zeros((H, W), dtype=bool))
        ft    = ft_masks.get(i, np.zeros((H, W), dtype=bool))

        orig = frame.copy()
        zs_panel = frame.copy()
        zs_panel[zs] = (zs_panel[zs]*0.5 + np.array([220,40,40])*0.5).clip(0,255).astype(np.uint8)
        ft_panel = frame.copy()
        ft_panel[ft] = (ft_panel[ft]*0.5 + np.array([40,80,220])*0.5).clip(0,255).astype(np.uint8)

        # Convert panels to BGR for cv2
        orig_bgr = cv2.cvtColor(orig,     cv2.COLOR_RGB2BGR)
        zs_bgr   = cv2.cvtColor(zs_panel, cv2.COLOR_RGB2BGR)
        ft_bgr   = cv2.cvtColor(ft_panel, cv2.COLOR_RGB2BGR)

        # Labels
        draw_label(orig_bgr, 'Original',               (50,  50,  50))
        draw_label(zs_bgr,   'Zero-Shot MedSAM2',      (30,  30, 180))
        draw_label(ft_bgr,   'ARCADE v2 Fine-Tuned',   (160, 60,  20))

        # Frame counter on each panel
        for panel in [orig_bgr, zs_bgr, ft_bgr]:
            draw_frame_counter(panel, i, n, prompt_frame)

        # White divider lines between panels
        divider = np.full((H, div, 3), 200, dtype=np.uint8)
        combined = np.concatenate([orig_bgr, divider, zs_bgr, divider, ft_bgr], axis=1)
        writer.write(combined)

    writer.release()
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'Saved: {out_path}  ({n} frames @ {fps}fps, {size_mb:.1f} MB)')

make_comparison_video(study1_frames, zs_masks_1, ft_masks_1, pf1,
                      f'{OUT_DIR}/study1_comparison.mp4', fps=12)

make_comparison_video(study2_frames, zs_masks_2, ft_masks_2, pf2,
                      f'{OUT_DIR}/study2_comparison.mp4', fps=12)

subprocess.run(['gsutil','-m','cp',
    f'{OUT_DIR}/study1_comparison.mp4',
    f'{OUT_DIR}/study2_comparison.mp4',
    f'{BUCKET}/results/video_eval/'], check=True)

print(f'All results at: {BUCKET}/results/video_eval/')